# CordonEnv 行为检测

用于验证 `DeepACO_implement.cordon_environment.CordonEnv` 是否符合你提出的动作规则。

In [44]:
import networkx as nx
import cordon_environment
import importlib
importlib.reload(cordon_environment)
from cordon_environment import CordonEnv

In [47]:
# 构造一个简单道路网络
G = nx.Graph()
G.add_edges_from([(0, 1), (1, 2), (2, 3), (1, 4)])

env = CordonEnv(G, max_steps=10, virtual_node=10000000)
state = env.reset()
print('initial state:', state)
print('initial actions:', env.available_actions())

initial state: [10000000]
initial actions: [0, 1, 2, 3, 4]


In [48]:
# 规则1：从虚拟节点 M 出发，首步可达任意真实节点
expected_first = set(G.nodes())
actual_first = set(env.available_actions())
print(actual_first)
# 规则2：选择节点 x 后，可选动作 = 邻居(M) ∪ 邻居(x)，且不能重复访问
x = 1
out = env.step(x)
print(out)
expected = (set(G.nodes()) | set(G.neighbors(x))) - {-1, x}
actual = set(env.available_actions())
print('state after step to 1:', out.state)


# 规则3：状态按访问顺序记录，不能重复访问
next_node = 2
out2 = env.step(next_node)
print(out2)
print('state after step to 2:', out2.state)
assert out2.state == [10000000, 1, 2]
assert 1 not in env.available_actions()
assert 2 not in env.available_actions()
print('✅ Rule-3 passed')

{0, 1, 2, 3, 4}
StepResult(state=[10000000, 1], reward=None, done=False, info={'current_node': 1, 'available_actions': [0, 2, 3, 4, 10000000]})
state after step to 1: [10000000, 1]
StepResult(state=[10000000, 1, 2], reward=None, done=False, info={'current_node': 2, 'available_actions': [0, 3, 4, 10000000]})
state after step to 2: [10000000, 1, 2]
✅ Rule-3 passed
